### Install libraries

In [36]:
pip install pandas numpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. Imports

In [2]:
import pandas as pd
import numpy as np
from collections import Counter

### 2. Load Data

In [3]:
from pathlib import Path

base_path = Path.cwd().parent

chapters = pd.read_csv(base_path / 'Data' / 'chapters.csv')
interactions = pd.read_csv(base_path / 'Data' / 'interactions.csv')


print("Chapters shape:", chapters.shape)
print("Interactions shape:", interactions.shape)


Chapters shape: (50000, 6)
Interactions shape: (1000000, 3)


In [5]:
# check top 5 rows
chapters.head()

,chapter_id,chapter_sequence_no,book_id,author_id,published_date,tags
0,2812946,1,139726,66847,22-03-1990,Fantasy|Horror
1,4330764,2,139726,66847,09-04-1990,Fantasy|Young Adult|Literary Fiction
2,2664499,3,139726,66847,07-04-1990,Fantasy
3,2260666,4,139726,66847,18-05-1990,Literary Fiction|Fantasy
4,6069976,1,191772,62262,30-07-2008,Horror|Young Adult|Romance|Graphic Novel


In [6]:
interactions.head()

,user_id,chapter_id,book_id
0,user_2378720,5894067,444295
1,user_2321122,2532511,785684
2,user_2335775,6777764,999595
3,user_7906001,7366896,748410
4,user_9981689,7853186,418083


IF you caught up with any error above, try with file paths directly and by uncommenting the code below

In [7]:
# path1 = r'C:\Users\vamsh\Documents\pratilipi_2\Data\chapters.csv'
# path2 = r'C:\Users\vamsh\Documents\pratilipi_2\Data\interactions.csv'
# chapters = pd.read_csv(path1)
# interactions = pd.read_csv(path2)

# print("Chapters shape:", chapters.shape)
# print("Interactions shape:", interactions.shape)

# chapters.head()
# interactions.head()

### 3.Preprocessing

3.1 Clean Tags

In [8]:
chapters['tags'] = chapters['tags'].fillna('').apply(lambda x: x.split('|'))

3.2 Merge Data

In [9]:
data = interactions.merge(chapters, on=['chapter_id', 'book_id'], how='left')

3.3 Sort Data

In [10]:
data = data.sort_values(['user_id', 'book_id', 'chapter_sequence_no'])

Checking the table after merge and sort

In [11]:
data.head()

,user_id,chapter_id,book_id,chapter_sequence_no,author_id,published_date,tags
541424,user_0000013,4485685,200976,5,97819,02-02-2011,[Thriller]
634433,user_0000013,1958839,461083,1,34356,17-04-2017,"[Adventure, Graphic Novel]"
620863,user_0000013,1646518,469844,2,52899,24-09-2016,"[Graphic Novel, Adventure, Mystery, Humor]"
541902,user_0000013,9045755,635456,7,46822,21-06-2016,[Historical Fiction]
540525,user_0000013,2055685,878246,1,22675,12-10-2016,[Romance]


### 4. Build Next Chapter Mapping


In [12]:
# Sort chapters correctly
book_chapters = chapters.sort_values(['book_id', 'chapter_sequence_no'])

# Create next chapter column
book_chapters['next_chapter'] = (
    book_chapters.groupby('book_id')['chapter_id']
    .shift(-1)
)

# Create mapping: chapter_id → next_chapter_id
next_chapter_map = dict(zip(
    book_chapters['chapter_id'],
    book_chapters['next_chapter']
))

### 5. Validation Check

In [13]:
print("Sample mapping:")
for k in list(next_chapter_map.keys())[:5]:
    print(k, "->", next_chapter_map[k])

Sample mapping:
5011702 -> 3515069.0
3515069 -> 6062471.0
6062471 -> nan
1727513 -> nan
4410559 -> 1591522.0


Checking last chapter handling

In [14]:
sample_book = chapters['book_id'].iloc[0]

book_data = chapters[chapters['book_id'] == sample_book] \
                .sort_values('chapter_sequence_no')

last_chapter = book_data.iloc[-1]['chapter_id']

print("Last chapter:", last_chapter)
print("Next chapter should be None:", next_chapter_map[last_chapter])

Last chapter: 2260666
Next chapter should be None: nan


### 6. Prediction Function

In [15]:
def predict_next_chapter(chapter_id):
    next_ch = next_chapter_map.get(chapter_id)
    
    if pd.isna(next_ch):
        return None
    
    return int(next_ch)

### 7. Popular Book Recommendation

In [16]:
book_popularity = interactions['book_id'].value_counts()

def recommend_popular_books(top_n=5):
    return book_popularity.head(top_n).index.tolist()

### 8.Tag-Based Recommendation

In [17]:
user_tags = data.groupby('user_id')['tags'].sum()
user_tags

user_id
user_0000013    [Thriller, Adventure, Graphic Novel, Graphic N...
user_0000115    [Graphic Novel, Fantasy, Romance, Historical F...
user_0000177    [Historical Fiction, Horror, Humor, Paranormal...
user_0000188    [Paranormal, Dystopian, Fantasy, Young Adult, ...
user_0000257    [Science Fiction, Fantasy, Paranormal, Mystery...
                                      ...                        
user_9999688    [Science Fiction, Paranormal, Humor, Paranorma...
user_9999862    [Science Fiction, Graphic Novel, Horror, Liter...
user_9999878    [Dystopian, Young Adult, Fantasy, Paranormal, ...
user_9999945    [Literary Fiction, Humor, Paranormal, Graphic ...
user_9999976    [Historical Fiction, Romance, Graphic Novel, M...
Name: tags, Length: 149803, dtype: object

In [18]:
def get_top_tags(tags_list, top_n=5):
    return [tag for tag, _ in Counter(tags_list).most_common(top_n)]

### Book Tag Aggregation

In [19]:
book_tags = chapters.groupby('book_id')['tags'].sum()

book_tags

book_id
100089         [Fantasy, Science Fiction, Fantasy, Fantasy]
100096                                            [Romance]
100193    [Thriller, Literary Fiction, Science Fiction, ...
100205    [Horror, Adventure, Adventure, Science Fiction...
100301          [Dystopian, Humor, Humor, Humor, Dystopian]
                                ...                        
999529    [Mystery, Thriller, Mystery, Mystery, Young Ad...
999595    [Historical Fiction, Literary Fiction, Crime, ...
999596    [Romance, Dystopian, Young Adult, Romance, Rom...
999876    [Romance, Young Adult, Adventure, Dystopian, H...
999910    [Dystopian, Historical Fiction, Horror, Dystop...
Name: tags, Length: 9575, dtype: object

### Recommendation Function by Tag

In [20]:
def recommend_by_tags(user_id, top_n=5):
    
    if user_id not in user_tags:
        return recommend_popular_books(top_n)
    
    tags = get_top_tags(user_tags[user_id])
    
    # ✅ Get books the user has already read
    seen_books = set(data[data['user_id'] == user_id]['book_id'].tolist())
    
    scores = []
    
    for book, b_tags in book_tags.items():
        # ✅ Skip already-read books
        if book in seen_books:
            continue
        score = len(set(tags) & set(b_tags))
        if score > 0:  # ✅ Skip completely unrelated books
            scores.append((book, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    
    return [book for book, _ in scores[:top_n]]

### Cold start for new books

In [21]:
interacted_books = set(interactions['book_id'].unique())

# Books that exist in metadata but have NO interactions → these are "new books"
all_books = set(chapters['book_id'].unique())
new_books = all_books - interacted_books

print(f"Total books: {len(all_books)}")
print(f"Books with interactions: {len(interacted_books)}")
print(f"New books (cold start): {len(new_books)}")

Total books: 9575
Books with interactions: 9575
New books (cold start): 0


In [22]:
def recommend_new_books_by_tags(query_tags, top_n=5):
    """
    Given a list of tags, recommend new books (zero interactions)
    that best match those tags.
    Used for surfacing freshly added books to relevant users.
    """
    scores = []

    for book_id, b_tags in book_tags.items():

        # ✅ Only consider books with no interactions
        if book_id not in new_books:
            continue

        score = len(set(query_tags) & set(b_tags))

        if score > 0:
            scores.append((book_id, score))

    scores.sort(key=lambda x: x[1], reverse=True)

    return [book for book, _ in scores[:top_n]]

In [23]:
# Test it — recommend new books for a thriller fan
query = ['THRILLER', 'CRIME']
result = recommend_new_books_by_tags(query, top_n=5)

print("New book recommendations for tags", query)
print(result)

New book recommendations for tags ['THRILLER', 'CRIME']
[]


### Smart Route Recommendation

In [24]:
def smart_recommend(user_id):
    
    user_data = data[data['user_id'] == user_id]
    
    # Case 1: New user — cold start
    if user_data.empty:
        return {
            "type": "new_user",
            "recommendations": recommend_popular_books()
        }
    
    # Get the last chapter read per book
    last_per_book = (
        user_data.sort_values('chapter_sequence_no')
                 .groupby('book_id')
                 .last()
                 .reset_index()[['book_id', 'chapter_id']]
    )
    
    continue_reading = []
    new_book_recommendations = []
    
    for _, row in last_per_book.iterrows():
        next_ch = predict_next_chapter(row['chapter_id'])
        
        if next_ch:
            # Book still in progress
            continue_reading.append({
                "book_id": row['book_id'],
                "next_chapter": next_ch
            })
        else:
            # Book completed → immediately recommend a replacement
            replacement = recommend_by_tags(user_id, top_n=3)
            new_book_recommendations.append({
                "completed_book": row['book_id'],
                "suggested_next_books": replacement
            })
    
    # Case 2: Has books in progress + some completed
    if continue_reading:
        return {
            "type": "reading_in_progress",
            "books_in_progress": continue_reading,
            "new_book_suggestions": new_book_recommendations  # can be empty list if none completed
        }
    
    # Case 3: Every single book is completed
    return {
        "type": "all_books_completed",
        "recommendations": recommend_by_tags(user_id)
    }

### Cold start testing

In [53]:
unknown = "vamshigaddi"
result = smart_recommend(unknown)

print("User:", unknown)
print("Recommendation:", result)

User: vamshigaddi
Recommendation: {'type': 'new_user', 'recommendations': [672233, 615276, 760434, 756944, 395485]}


### Next chapter Recommendation

In [51]:
sample_user = data['user_id'].iloc[0]

result = smart_recommend(sample_user)

print("User:", sample_user)
print("Recommendation:", result)

User: user_0000013
Recommendation: {'type': 'reading_in_progress', 'books_in_progress': [{'book_id': 200976, 'next_chapter': 2905074}, {'book_id': 461083, 'next_chapter': 5948621}, {'book_id': 635456, 'next_chapter': 6667201}, {'book_id': 878246, 'next_chapter': 7960266}, {'book_id': 964446, 'next_chapter': 4637663}, {'book_id': 976042, 'next_chapter': 7069830}], 'new_book_suggestions': [{'completed_book': 469844, 'suggested_next_books': [101429, 101655, 101760]}]}


In [52]:
for user in data['user_id'].unique()[:5]:
    print("\nUser:", user)
    print(smart_recommend(user))


User: user_0000013
{'type': 'reading_in_progress', 'books_in_progress': [{'book_id': 200976, 'next_chapter': 2905074}, {'book_id': 461083, 'next_chapter': 5948621}, {'book_id': 635456, 'next_chapter': 6667201}, {'book_id': 878246, 'next_chapter': 7960266}, {'book_id': 964446, 'next_chapter': 4637663}, {'book_id': 976042, 'next_chapter': 7069830}], 'new_book_suggestions': [{'completed_book': 469844, 'suggested_next_books': [101429, 101655, 101760]}]}

User: user_0000115
{'type': 'reading_in_progress', 'books_in_progress': [{'book_id': 124851, 'next_chapter': 4090334}, {'book_id': 475202, 'next_chapter': 2840330}, {'book_id': 611133, 'next_chapter': 9879460}, {'book_id': 648583, 'next_chapter': 9516166}, {'book_id': 834928, 'next_chapter': 5148182}], 'new_book_suggestions': [{'completed_book': 816137, 'suggested_next_books': [102146, 106945, 108739]}]}

User: user_0000177
{'type': 'reading_in_progress', 'books_in_progress': [{'book_id': 239794, 'next_chapter': 2612479}, {'book_id': 2644

### Recommending a new book to user based on his preferences


In [20]:
user_id = 'user_0000013' ## thriller

#101429 - ADVENTURE,HISTORICAL FICTION
# 101655 - THRILLER,CRIME
#101760 - THRILLER,ADEVENTURE
result = recommend_by_tags(user_id)

print("User:", user_id)
print("Recommendation:", result)

User: user_0000013
Recommendation: [101429, 101655, 101760, 103234, 104539]


In [41]:
correct = 0
total = 0

for _, row in data.iterrows():
    pred = predict_next_chapter(row['chapter_id'])
    actual = next_chapter_map.get(row['chapter_id'])
    if pd.isna(actual):
        actual = None
    
    if pred == actual:
        correct += 1
    
    total += 1

print("Next Chapter Accuracy:", correct / total)

Next Chapter Accuracy: 1.0


saving data into csv file


In [24]:
data.to_csv("final_data.csv")

#### Recommendation Evaluation

In [21]:
user_books = interactions.groupby('user_id')['book_id'].apply(list)

In [46]:
import random
from collections import Counter

def recall_at_k(k=5, sample_size=500):
    random.seed(42)
    hits = 0
    total = 0
    
    sampled_users = random.sample(
        list(user_books.items()),
        min(sample_size, len(user_books))
    )
    
    for user, books in sampled_users:
        
        if len(books) < 2:
            continue
        
        # Hide one book
        test_book = random.choice(books)
        train_books = [b for b in books if b != test_book]
        
        if not train_books:
            continue
        
        # Build weighted tag profile
        temp_tags = []
        for b in train_books:
            temp_tags.extend(book_tags.get(b, []))
        
        tag_counts = Counter(temp_tags)  # ⭐ important
        
        # Score books
        scores = []
        for book, b_tags in book_tags.items():
            
            # Skip books already seen
            if book in train_books:
                continue
            
            # Weighted score instead of simple overlap
            score = sum(tag_counts[tag] for tag in b_tags if tag in tag_counts)
            
            if score > 0:  # ignore completely unrelated books
                scores.append((book, score))
        
        if not scores:
            continue
        
        # Sort by score
        scores.sort(key=lambda x: x[1], reverse=True)
        recommended_books = [b for b, _ in scores[:k]]
        
        # Check hit
        if test_book in recommended_books:
            hits += 1
        
        total += 1
    
    return hits / total if total > 0 else 0

In [47]:
print("Recall@5 (sample):", recall_at_k(k=10, sample_size=500))

Recall@5 (sample): 0.01002004008016032


The Recall@K is relatively low because the model relies on tag-based similarity, which is a weak and noisy signal. It does not capture deeper user behavior or relationships between books.

To improve this, I would incorporate collaborative filtering, add popularity signals, and use a better ranking mechanism to prioritize relevant books.